## Download movielens

In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
ml_path = "/home/caio/dev/dynamicTasteDistortion/data/movielens/ml_1m.pkl"

In [3]:
ml = pd.read_pickle(ml_path)

In [4]:
ml

,item,genres,user,rating,timestamp,binarized_rating
0,1,"[animation, children's, comedy]",1,5,978824268,1
1,1,"[animation, children's, comedy]",6,4,978237008,1
2,1,"[animation, children's, comedy]",8,4,978233496,1
3,1,"[animation, children's, comedy]",9,5,978225952,1
4,1,"[animation, children's, comedy]",10,5,978226474,1
...,...,...,...,...,...,...
1000204,3952,"[drama, thriller]",5812,4,992072099,1
1000205,3952,"[drama, thriller]",5831,3,986223125,0
1000206,3952,"[drama, thriller]",5837,4,1011902656,1
1000207,3952,"[drama, thriller]",5927,1,979852537,0


In [5]:

# Dropando por enquanto. Se eu quiser usar quantidade de interações como sinal de relevância, funcionaria.
interaction_matrix = ml[["user", "item"]]

In [6]:
interaction_matrix

,user,item
0,1,1
1,6,1
2,8,1
3,9,1
4,10,1
...,...,...
1000204,5812,3952
1000205,5831,3952
1000206,5837,3952
1000207,5927,3952


### Preprocessing

Just standardizing ids

In [7]:
interaction_matrix.loc[:, 'userIdx'] = interaction_matrix['user'].astype('category').cat.codes
interaction_matrix.loc[:, 'itemIdx'] = interaction_matrix['item'].astype('category').cat.codes

df = interaction_matrix.loc[:, ['userIdx', 'itemIdx']]

/tmp/ipykernel_27085/1292339781.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_matrix.loc[:, 'userIdx'] = interaction_matrix['user'].astype('category').cat.codes
/tmp/ipykernel_27085/1292339781.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_matrix.loc[:, 'itemIdx'] = interaction_matrix['item'].astype('category').cat.codes


In [8]:
df

,userIdx,itemIdx
0,0,0
1,4,0
2,6,0
3,7,0
4,8,0
...,...,...
1000204,5080,3700
1000205,5096,3700
1000206,5102,3700
1000207,5183,3700


In [9]:
df.userIdx.min(), df.userIdx.max(), df.userIdx.nunique()

(0, 5288, 5289)

## Build similarity matrix

Because in our problem we are using implicit feedback, we need a similarity measure such as Jaccard index for it. We'll simply calculate the intersection of interacted items

In [10]:
def jaccard_similarity_matrix(interaction_tensor, user_based=True):
    """
    Computes full pairwise Jaccard similarity matrix.
    Returns (n_users, n_users) if user_based, else (n_items, n_items).
    """
    M = interaction_tensor if user_based else interaction_tensor.T
    M = M.float()

    intersection = M @ M.T                       #I[i,j] represents the ammount of shared interactions between users i and j (likewise for user_Based=false, for items)
    row_sums = M.sum(dim=1)                      # (N,) items/users per row
    row_sums_i = row_sums.unsqueeze(1)  # (N, 1), broadcasts across columns
    row_sums_j = row_sums.unsqueeze(0)  # (1, N), broadcasts across rows
    union = row_sums_i + row_sums_j - intersection # (N, 1) + (N,1) broadcasted to (N, N)

    similarity = torch.zeros_like(intersection)
    nonzero_mask = union > 0
    similarity[nonzero_mask] = intersection[nonzero_mask] / union[nonzero_mask]

    return similarity

## high level functionalities


In [11]:
df

,userIdx,itemIdx
0,0,0
1,4,0
2,6,0
3,7,0
4,8,0
...,...,...
1000204,5080,3700
1000205,5096,3700
1000206,5102,3700
1000207,5183,3700


In [12]:
def build_interaction_tensor_from_df(df):
    # we assume that both idx columns are standardized and continuous
    n_users = df.userIdx.max() + 1
    n_items = df.itemIdx.max() + 1
    matrix = torch.zeros(size=(n_users, n_items))
    coordinates = df.values
    rows = coordinates[:, 0]
    cols = coordinates[:, 1]
    matrix[rows, cols] = 1
    return matrix


In [13]:
interaction_matrix = build_interaction_tensor_from_df(df)

In [14]:
sim_matrix = jaccard_similarity_matrix(interaction_matrix)

In [30]:
def get_batch_neighborhoods(users, similarity, k=10):
    sims = similarity[users].clone()
    # ingore main diagonal
    sims[torch.arange(len(users)), users] = -1
    return torch.topk(sims, k).indices

In [28]:
def score_matrix_loop_k(users, items, similarity_matrix, interaction_matrix, k=10):

    # (n_u, k)
    neighbors = get_batch_neighborhoods(users, similarity_matrix, k) 
    # (n_u, k)
    sims = similarity_matrix[users.unsqueeze(1), neighbors]

    sum_all_sims = sims.sum(dim=1)  # (n_u,)
    sum_filtered_sims = torch.zeros(len(users), len(items))  # (n_u, n_i)

    for i in range(k):
        neighbor_i = neighbors[:, i]                              # (n_u,)
        sim_i = sims[:, i]                                        # (n_u,)
        # check if each neighbor has interacted with the candidate items (n_u, n_i).
        interacted_i = interaction_matrix[neighbor_i][:, items].bool() 
        # masks out neighbors without interaction for the particular item.
        sum_filtered_sims += sim_i.unsqueeze(1) * interacted_i

    scores = torch.where(
        sum_all_sims.unsqueeze(1) > 0,
        sum_filtered_sims / sum_all_sims.unsqueeze(1),
        torch.zeros_like(sum_filtered_sims)
    )
    return scores

In [21]:
n_users = df.userIdx.max() + 1
n_items = df.itemIdx.max() + 1

In [25]:
users = torch.arange(0, n_users)
items = torch.arange(0, n_items)


In [33]:
score_matrix_loop_k(users, items, sim_matrix, interaction_matrix)

tensor([[0.9117, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3040, 0.2032, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.2991, 0.0946, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.5977, 0.0962, 0.0000,  ..., 0.0000, 0.0000, 0.0984],
        [0.6061, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.8006, 0.1011, 0.0978,  ..., 0.0000, 0.0000, 0.0972]])

In [35]:
def recommend(users, items, similarity_matrix, interaction_matrix, k=10, top_n=None):
    scores = score_matrix_loop_k(users, items, similarity_matrix, interaction_matrix, k)  # (B_u, B_i)

    if top_n is None:
        top_n = len(items)  # return everything, just sorted

    top_scores, top_positions = torch.topk(scores, top_n, dim=1)
    top_item_ids = items[top_positions]

    return top_item_ids, top_scores

In [39]:
recommend(users, items, sim_matrix, interaction_matrix, top_n=100)[0]

tensor([[ 581,    0, 2893,  ...,  576,  583,  346],
        [2155, 1843,  370,  ...,  742, 1273, 1929],
        [1106, 1104,  466,  ..., 1293, 1816,  574],
        ...,
        [ 593, 1158, 1137,  ...,    0,  863,  888],
        [1762, 2363,  856,  ..., 1023, 1892, 1097],
        [ 712, 1767,  860,  ..., 3261,  861, 2048]])

In [26]:
users

tensor([   0,    1,    2,  ..., 5286, 5287, 5288])

In [ ]:
score_matrix_loop_k

In [19]:
score(1, 1, sim_matrix, interaction_matrix)

0.20321889221668243